In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from sklearn.preprocessing import OrdinalEncoder

### Data Analysis

In [144]:
arbres = pd.read_csv("train.csv", index_col = 'ID_ARBRE')
X = arbres[["type_sol", "classe_age", "classe_hauteur", "classe_circonference", "port_arbre", "vigueur_pousse", "plaie_collet", "fissure_tronc", "plaie_tronc", "fissure_houppier", "bois_mort_houppier", "plaie_houppier", "esperance_maintien"]].copy()
X["esperance_maintien"] = X["esperance_maintien"].astype(str)
y = arbres[["classification_diagnostic"]]

# Prev : 
prev = pd.read_csv("prev.csv", index_col = 'ID_ARBRE')
prev = prev[["type_sol", "classe_age", "classe_hauteur", "classe_circonference", "port_arbre", "vigueur_pousse", "plaie_collet", "fissure_tronc", "plaie_tronc", "fissure_houppier", "bois_mort_houppier", "plaie_houppier", "esperance_maintien"]].copy()
prev["esperance_maintien"] = prev["esperance_maintien"].astype(str)

In [146]:
X.head()

,type_sol,classe_age,classe_hauteur,classe_circonference,port_arbre,vigueur_pousse,plaie_collet,fissure_tronc,plaie_tronc,fissure_houppier,bois_mort_houppier,plaie_houppier,esperance_maintien
ID_ARBRE,,,,,,,,,,,,,
0,MA,A,H1,C1,R5,P,RCPPL,TPF,TPLS,HPF,HPBM,HPLS,1.0
1,MA,A,H2,C3,R5,P,RCPLS,TFO,TPLNC,HFO,HPBM,HPLNC,2.0
2,GR,A,H1,C2,R5,P,RCPPL,TPF,TPLC,HPF,HPBM,HPLS,1.0
3,Gr,A,H2,C3,R5,P,RCPLC,TPF,TPLC,HPF,HPBM,HPLS,1.0
4,MA,JA,H1,C1,L,P,RCPPL,TPF,TPLS,HPF,HPBM,HPPL,1.0


### Decision Tree (OOP method)

In [150]:
class Node:

    def __init__(self, feature, value, cost, left_node, right_node, depth, pop, prediction=None):
        self.feature = feature # Un noeud contient une feature et une valeur pour split notre dataset
        self.value = value
        self.left_node = left_node # Il contient également une branche gauche...
        self.right_node = right_node #... et droite, appelée ici "node"
        self.cost = cost # le cout de la séparation calculé avec l'indice Gini
        self.depth = depth # Profondeur de l'arbre
        self.pop = pop # Taille de l'echantillon du noeud
        self.prediction = prediction #La classe majoritaire du noeud 


class decision_tree_classifier :

    def __init__(self, max_depth, seuil, min_samples_split, min_impurity_decrease):
        self.max_depth = max_depth #La profondeur maximale
        self.seuil = seuil #Le seuil de score Gini minimal pour split notre dataset
        self.min_samples_split = min_samples_split
        self.min_impurity_decrease = min_impurity_decrease

    def _split(self, feature, value, dataset):
        left = dataset[dataset[feature] == value]
        right = dataset[dataset[feature] != value]
        return left, right

    def _entropie(self, y):
        counts = y.value_counts().to_numpy()
        population = np.sum(counts)
        proportions = counts / population
        # On évite log2(0) en ne considérant que p > 0
        entropy = -np.sum([p * np.log2(p) for p in proportions if p > 0])
        return entropy

    def _best_split(self, X, y):
        best_feature = X.columns[0]
        best_value = np.unique(X[X.columns[0]])[0] #On initialise la best feature comme étant la 1ère colonne
        best_cost = 1 # Score initialisé au plus bas : 1
        
        for feature in X.columns:
            for value in X[feature].unique():

                idx_left = X[feature] == value
                y_left = y[idx_left]
                y_right = y[~idx_left]
                
                # Évite une division par zéro
                if len(y_left) == 0 or len(y_right) == 0:
                    continue
                cost = (len(y_left) * self._entropie(y_left) + len(y_right) * self._entropie(y_right))/len(y)
                
                if cost < best_cost:
                    # Si le couple (feature + valeur) possède le coût le plus faible, alors on le sauvegarde.
                    best_cost = cost
                    best_value = value
                    best_feature = feature
        return best_feature, best_value, best_cost

    def fit(self, X=None, y=None, depth=0):
        # Condition d'arrêt de la récursion :
        
        # Pré-pruning : si le nombre d'exemples est inférieur à min_samples_split, arrêter
        if self.min_samples_split is not None and len(y) < self.min_samples_split:
            mode_values = y.mode()
            prediction = mode_values.iloc[0] if not mode_values.empty else None
            return Node(None, None, 0, None, None, depth, len(y), prediction=prediction)

        
        # Condition d'arrêt classique : profondeur maximale ou impureté suffisamment faible.
        if depth > self.max_depth or self._entropie(y) <= self.seuil:
            # Si y est vide ou on a atteint un état terminal, on retourne un noeud terminal
            if y.empty:
                return Node(None, None, 0, None, None, depth, 0, prediction=None)
            else:
                prediction = y.mode().iloc[0]
                return Node(None, None, 0, None, None, depth, len(y), prediction=prediction)
        
        # Recherche du meilleur split
        best_feature, best_value, best_cost = self._best_split(X, y) #On cherche la meilleure (feature + value) pour split
        if best_feature is None:
            prediction = y.mode().iloc[0]  # Classe majoritaire
            return Node(feature=None, value=None, cost=0, left_node=None, right_node=None, depth=depth, pop=len(y), prediction=prediction)

        # Vérifier si l'amélioration est suffisante pour continuer (min_impurity_decrease)
        if self.min_impurity_decrease != None:
            
            parent_impurity = self._entropie(y)
            if parent_impurity - best_cost < self.min_impurity_decrease:
                if y.empty:
                    return Node(None, None, 0, None, None, depth, 0, prediction=None)
                prediction = y.mode().iloc[0]
                return Node(None, None, 0, None, None, depth, len(y), prediction=prediction)
        
        # On découpe selon l'index:
        idx_left = X[best_feature] == best_value
        X_left, y_left = X[idx_left], y[idx_left] 
        X_right, y_right = X[~idx_left], y[~idx_left]

        left_node = self.fit(X_left, y_left, depth + 1)
        right_node = self.fit(X_right, y_right, depth + 1)        

        #A chaque tour de fonction, on retourne un noeud contenant les 2 noeuds/branches suivants, et des informations sur le noeud actuel.
        return Node(feature=best_feature, value=best_value, cost=best_cost, left_node=left_node, right_node=right_node, depth=depth, pop=len(y), prediction=None)

    
    def predict_row(self, row, node=None):
        # predict() prend une ligne de dataset à la fois ("row") et la racine de l'arbre qu'on va parcourir pour prédire nos données.
        # Si notre noeud a pour paramètre feature = None, alors c'est qu'on fait face à un noeud terminal / feuille

        if node.feature is None:
            return node.prediction # Ici on retourne alors la classe majoritaire du noeud.
        if row[node.feature] == node.value:
            return self.predict_row(row, node.left_node)
        else:
            return self.predict_row(row, node.right_node)

    def predict(self, X, root):
        """
        Prédit les classes pour l'ensemble du DataFrame X.
        """
        return pd.DataFrame(X.apply(lambda row: self.predict_row(row, node=root), axis=1), columns=[y.columns[0]])

### Train_test_split - 0.90 / 0.20

In [153]:
from sklearn.model_selection import train_test_split

In [155]:
X_train, X_test, Y_train, Y_test = train_test_split(X, y, test_size=0.20, random_state=42)

### Entraînement modèle - StratifiedKFold

In [158]:
tree = decision_tree_classifier(
    max_depth=10, 
    seuil=0.2, 
    min_samples_split = 10, 
    min_impurity_decrease = 0.01
)
root = tree.fit(X_train, Y_train) 

In [160]:
test_predict = tree.predict(X_test, root)
test_predict.value_counts()

# On a passé à predict() notre node=root entrainé précédemment car c'est lui qui va nous permettre de trouver le 
# "chemin" de l'arbre que nos données ont à suivre pour prédire leur classe.

classification_diagnostic
C2                           56
C1                           53
Name: count, dtype: int64

### Accuracy

In [163]:
def are(predict, Y):
    correct = np.sum(np.array(predict).flatten() == Y.to_numpy().flatten())
    taux = np.round((correct / len(Y)) * 100, 5)  # Pourcentage de bonnes prédictions
    return taux

In [165]:
are(test_predict, Y_test)

77.06422

### Recherches des paramètres

In [65]:
from sklearn.model_selection import StratifiedKFold
from joblib import Parallel, delayed

In [75]:
depths_array = [10]
entropy_array = [0.10, 0.15, 0.20, 0.25, 0.30]
#min_samples_split_array = [2, 5, 10]
#min_impurity_decrease_array = [0.0, 0.01, 0.02, 0.05]

skf = StratifiedKFold(n_splits=7, shuffle=True, random_state=42)

def best_params(depth, seuil, folds, X, y):
    scores = []
    for train_index, test_index in folds.split(X, y):
        X_train, X_test = X.iloc[train_index], X.iloc[test_index]
        Y_train, Y_test = y.iloc[train_index], y.iloc[test_index]
        
        tree = decision_tree_classifier(max_depth=depth, 
                                        seuil=seuil, 
                                        min_samples_split=None,
                                        min_impurity_decrease=None
                                       )
        root = tree.fit(X_train, Y_train)
        predictions = tree.predict(X_test, root=root)
        scores.append(are(predictions, Y_test))
    mean_score = np.mean(scores)
    return {
        "depth": depth,
        "seuil": seuil,
        "mean_score": mean_score
    }

In [77]:
# On distribue les tâches aux différents coeurs du processeurs pour accélérer la recherche des meilleurs paramètres.

resultats = Parallel(n_jobs=-1)(
    delayed(best_params)(depth, seuil, skf, X, y)
    for depth in depths_array
    for seuil in entropy_array
    #for mss in min_samples_split_array
    #for mid in min_impurity_decrease_array
)

In [79]:
results_df = pd.DataFrame(resultats)
results_df.sort_values(by="mean_score", ascending=False)

,depth,seuil,mean_score
3,10,0.25,77.755579
4,10,0.30,77.755579
2,10,0.20,77.572429
0,10,0.10,77.206129
1,10,0.15,77.206129


### Prédictions sur Prev

In [580]:
prev_predictions = tree.predict(prev, root=root)

In [584]:
prev_predictions.value_counts()

classification_diagnostic
C2                           77
C1                           54
C3                            5
C5                            1
Name: count, dtype: int64

In [565]:
prev_predictions.to_csv("prev_dpth10_gni025.csv")

### Submition

In [541]:
sub = prev_predictions
sub.index.name = "ID_ARBRE"
id_arbre = prev_trie.index  # On récupère l'index chez x2
sub["ID_ARBRE"] = id_arbre  # Ajoute l'index comme colonne
sub = sub.set_index("ID_ARBRE")  # On définie 'ID_ARBRE' en tant qu'index

In [528]:
sub.to_csv("sub_dpth10_gni025.csv")

NameError: name 'sub' is not defined